```sql
03 - Feature Iteration

Goal:
- start from the baseline setup
- add small feature groups one by one
- compare each iteration against the same validation split

This notebook is for feature testing, not for model tuning.
```

In [1]:
import pathlib as Path

In [2]:
import pandas as pd
import numpy as np

In [3]:
from lightgbm import LGBMClassifier

In [4]:
from sklearn.metrics import roc_auc_score, average_precision_score

#### Import Data

In [5]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [6]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [7]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [8]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [9]:
available_features = [c for c in base_features if c in train_df.columns]

In [10]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [11]:
new_features = ["log_txnamt", "has_identity"]

In [12]:
feature_cols=available_features+new_features

#### Model Training and Result Function

In [13]:
def model_input_data(train_df,valid_df,feature_cols,target_col="isFraud"):

    #Splitting Target vs Predictors
    X_train=train_df[feature_cols].copy()
    X_valid=valid_df[feature_cols].copy()

    y_train=train_df[target_col].copy()
    y_valid=valid_df[target_col].copy()

    #Seperating Categorical and Numerical Columns
    cat_cols=X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols=X_train.select_dtypes(include="number").columns.tolist()

    #Filling Missing Values
    for col in num_cols:
        X_train[col]=X_train[col].fillna(X_train[col].median())
        X_valid[col]=X_valid[col].fillna(X_valid[col].median())

    for col in cat_cols:
        X_train[col]=X_train[col].fillna("Missing").astype(str)
        X_valid[col]=X_valid[col].fillna("Missing").astype(str)
    
    # Typecasting all strings into category
    for col in cat_cols:
        X_train[col]=X_train[col].astype("category")
        X_valid[col]=X_valid[col].astype("category")
    
    return X_train, X_valid, y_train, y_valid, cat_cols

In [31]:
def model_results(train_df,valid_df,feature_cols,model_name="lgbm"):
    X_train, X_valid, y_train, y_valid, cat_cols=model_input_data(train_df,valid_df,feature_cols)
    model=LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=64,subsample=0.8,
                         colsample_bytree=0.8,random_state=42,class_weight="balanced")
    model.fit(X_train, y_train, categorical_feature=cat_cols)

    pred=model.predict_proba(X_valid)[:,1]

    results = {
        "n_features": len(feature_cols),
        "roc_auc": roc_auc_score(y_valid,pred),
        "pr_auc":average_precision_score(y_valid,pred),
        "pred":pred,
        "model":model
    }

    return results

#### Baseline Results

In [32]:
baseline_result = model_results(train_df, valid_df, feature_cols)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028607 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [33]:
print("Baseline ROC-AUC:", round(baseline_result["roc_auc"], 5))
print("Baseline PR-AUC :", round(baseline_result["pr_auc"], 5))

Baseline ROC-AUC: 0.80776
Baseline PR-AUC : 0.23062


#### Feature Set 1 : Adding columns for Missing

In [34]:
missing_cols=["DeviceType", "DeviceInfo", "P_emaildomain", "R_emaildomain"]

In [35]:
for col in missing_cols:
    if col in train_df.columns:
        train_df[f"{col}_missing"]=train_df[col].isna().astype(int)
        valid_df[f"{col}_missing"]=valid_df[col].isna().astype(int)

In [36]:
fs1_features=feature_cols+[new_col for new_col in train_df.columns if new_col.endswith("_missing")]

In [37]:
fs1_result=model_results(train_df, valid_df, fs1_features)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.029276 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1816
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [38]:
print("Feature Set 1 ROC-AUC:", round(fs1_result["roc_auc"], 5))
print("Feature Set 1 PR-AUC :", round(fs1_result["pr_auc"], 5))

Feature Set 1 ROC-AUC: 0.80973
Feature Set 1 PR-AUC : 0.233


In [39]:
####Both metrics have slightly improved over baseline

#### Feature Set 2: Creating Buckets for Amount Fields

In [40]:
for df_ in [train_df,valid_df]:
    df_["txn_amt_bin"]=pd.cut(df_["TransactionAmt"],
               bins=[-1,10,50,100,500,np.inf],
               labels=["0_10","10_50","50_100","100_500","500_Plus"]).astype(str)

In [41]:
fs2_features=feature_cols+["txn_amt_bin"]

In [42]:
fs2_result=model_results(train_df, valid_df, fs2_features)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022741 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1814
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [43]:
print("Feature Set 1 ROC-AUC:", round(fs2_result["roc_auc"], 5))
print("Feature Set 1 PR-AUC :", round(fs2_result["pr_auc"], 5))

Feature Set 1 ROC-AUC: 0.80984
Feature Set 1 PR-AUC : 0.23123


In [44]:
#Both metrics have slightly improved over baseline

In [45]:
baseline_result

{'n_features': 16,
 'roc_auc': 0.8077605483099215,
 'pr_auc': 0.2306206204200023,
 'pred': array([0.8401635 , 0.00363045, 0.48440845, ..., 0.03192274, 0.19222926,
        0.44757369], shape=(118108,)),
 'model': LGBMClassifier(class_weight='balanced', colsample_bytree=0.8,
                learning_rate=0.05, n_estimators=300, num_leaves=64,
                random_state=42, subsample=0.8)}

#### Comparing the results

In [46]:
df_comp=pd.DataFrame([
    {
        "version":"baseline",
        "n_feature":baseline_result["n_features"],
        "roc_auc":baseline_result["roc_auc"],
        "pr_auc":baseline_result["pr_auc"],
    },
    {
        "version":"fs1",
        "n_feature":fs1_result["n_features"],
        "roc_auc":fs1_result["roc_auc"],
        "pr_auc":fs1_result["pr_auc"],
    },
    {
        "version":"fs2",
        "n_feature":fs2_result["n_features"],
        "roc_auc":fs2_result["roc_auc"],
        "pr_auc":fs2_result["pr_auc"],
    }
])

In [47]:
df_comp.sort_values(["pr_auc", "roc_auc"], ascending=False)

,version,n_feature,roc_auc,pr_auc
1,fs1,20,0.809733,0.232999
2,fs2,17,0.809842,0.231228
0,baseline,16,0.807761,0.230621


(In Progress)

```sql
Outcome

In this notebook we are able to achieve:
- a baseline benchmark
- one or two simple feature waves tested on the same split
- a comparison table showing what actually helped
```